In [3]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('scores/S1/imagined_speech/gpt_words_5/alpha_repeat-1.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['story_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('alpha_repeat-1', 'WER'): np.float64(1.704767221094964), ('alpha_repeat-1', 'BLEU'): np.float64(1.7702071204224945), ('alpha_repeat-1', 'METEOR'): np.float64(1.0183319350579307), ('alpha_repeat-1', 'BERT'): np.float32(2.8672938)}


In [4]:
window_zscores = {'gpt_layer': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for gpt_layer in [3,4,5,8,10]:
    for task in ['alpha_repeat-1']:
        scores = np.load(f'scores/S1/imagined_speech/gpt_words_{gpt_layer}/{task}.npz', allow_pickle=True)['window_zscores'].item()
        # print(scores['window_zscores'].item())
        window_zscores['gpt_layer'].append(gpt_layer)
        window_zscores['WER'].append(scores[(task, 'WER')])
        window_zscores['BLEU'].append(scores[(task, 'BLEU')])
        window_zscores['METEOR'].append(scores[(task, 'METEOR')])
        window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores

{'gpt_layer': [3, 4, 5, 8, 10],
 'WER': [array([ 1.69808573,  1.38775688,  1.67618205,  1.585472  ,  2.07433026,
          2.12003682,  2.17737349,  1.35443391,  1.96127249, -0.03419428,
          1.4057339 ,  0.60253839,  0.91107536,  1.30702858,  0.94210129,
          2.08681939,  1.74927147,  1.10228352,  1.87232484,  1.10833761,
          1.16980045,  1.94580336,  1.54647383,  0.89301756,  1.08875183,
          1.63041194,  1.67835275,  2.21474418,  2.62475604,  2.75489   ,
          2.93921443,  2.97099294,  2.97968115,  3.17031994,  3.64385924,
          2.17547276,  0.62178464,  0.62662831,  0.53665631,  0.64655087,
          0.59646131]),
  array([ 3.11931129e-16, -4.30414027e-01, -8.21994937e-01, -5.32362316e-02,
          4.64056150e-01,  4.69522575e-01,  5.41676263e-01,  5.79982503e-01,
         -4.45849264e-01, -8.55027539e-01,  4.42524676e-01, -2.17194657e-01,
          9.60185512e-02, -2.90502645e-01,  1.18236962e-01, -4.55169639e-02,
          4.23338246e-01,  4.31988660

In [5]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(563*3)
m=1689

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(3):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

# S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
# S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
# S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
# print(S1,S2,S3,'BERT')

results_df

41
41
41
=
1689


,gpt_layer,WER,BLEU,METEOR,BERT
0,3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,4,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, ..."
3,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,10,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [8]:
results_df

,gpt_layer,WER,BLEU,METEOR,BERT
0,6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ..."
3,9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,10,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [6]:
gpt_layer_6=np.array(results_df.loc[0, 'BERT']).mean()
gpt_layer_7=np.array(results_df.loc[1, 'BERT']).mean()
gpt_layer_8=np.array(results_df.loc[2, 'BERT']).mean()
gpt_layer_9=np.array(results_df.loc[3, 'BERT']).mean()
gpt_layer_10=np.array(results_df.loc[4, 'BERT']).mean()
to_file = pd.DataFrame({'gpt_layer':[3,4,5,8,10], 'significantly_decoded': [gpt_layer_6,gpt_layer_7,gpt_layer_8,gpt_layer_9,gpt_layer_10]})
to_file.to_csv('imagined_speech_gpt_words.csv', index=False)

to_file

,gpt_layer,significantly_decoded
0,3,0.000000
1,4,0.000000
2,5,0.317073
3,8,0.365854
4,10,0.000000
